# Transfer learning through adversarial domain-invariant learning
### A small autoencoder: MNIST → USPS

**Question:** can a representation learned using labelled MNIST images classify USPS images when USPS digit labels are hidden during training?

| Digit demonstration | PBSHM analogy |
|---|---|
| MNIST / USPS | Source / target structure or population |
| Digit identity, 1–9 | Shared health or damage class |
| Handwriting, resolution, acquisition | Structure, sensor and environmental differences |
| Shared encoder | Transferable feature representation |

This is a classification demonstration, not an SHM dataset or a novelty detector. We assume the same nine classes in both domains. Digit **0 is excluded**.

We compare the same small architecture and training budget, with and without a domain-adversarial loss. The latent vector has **exactly two coordinates**, so we can plot it directly. No model has been trained for this repository; there are no claimed accuracy results.


## 1. Setup: train yourself or load your checkpoints

Install `requirements.txt` in your Python environment and select that environment as the notebook kernel. `MODE = "load"` is the safe default: it never falls back to training. To create both checkpoints yourself, set `MODE = "train"` and run the notebook. Then switch back to `"load"` for a lecture.

The default uses 12,000 randomly selected MNIST training images, all eligible USPS training images, batch size 128, and 20 epochs. These are starting settings, not tuned or validated performance settings. Both experiments start from the same random weights and receive the same MNIST batches. CPU is the default; actual runtime depends on your machine. The first data cells download the official datasets.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import os

import numpy as np
import torch
import matplotlib.pyplot as plt

from demo import (
    Config, Autoencoder, training_data, test_data, fit,
    save_checkpoint, load_checkpoint, evaluate,
    plot_inputs, plot_latent, plot_reconstructions,
)

MODE = "load"                  # "train" or "load"; no automatic fallback
DEVICE = "cpu"
DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints")
cfg = Config()
torch.set_num_threads(min(4, os.cpu_count() or 1))
assert MODE in {"train", "load"}
print(asdict(cfg))


## 2. Raw / input domain difference

MNIST images have 28 × 28 pixels; USPS images have 16 × 16 pixels. For the network, USPS is bilinearly resized to 28 × 28, and both datasets are scaled to [0, 1]. We add no artificial domain shift, augmentation or domain-specific normalization.

**Label boundary:** defining the requested 1–9 benchmark uses the dataset metadata once to remove digit 0 from both domains. This is a closed-set curation assumption, not a target learning signal. The target training wrapper then stores **images only**; it cannot return digit labels. Selecting the target subset, if requested, is random, not class-balanced. Torchvision already provides labels in 0–9; we subtract one only after filtering to obtain classifier indices 0–8.

Only official training splits are opened here. Official test splits and their labels are opened after both models have been trained or loaded.


In [ ]:
source_train, target_train, native_examples = training_data(DATA_DIR, cfg)
print(f"Training images: MNIST {len(source_train):,}; USPS {len(target_train):,}")
print("Source item: (image, digit index); target item: image only")
plot_inputs(native_examples)
plt.show()


## 3. Architecture and learning objectives

The encoder is `Conv(1→16) → ReLU → Pool → Conv(16→32) → ReLU → Pool → FC(64) → z(2)`.

Three branches use $z$:

- **Decoder:** `2 → 64 → 128 → 784`, with sigmoid output, reconstructs the image.
- **Health / digit classifier:** `2 → 32 → 9`, predicts the shared class.
- **Domain classifier:** `2 → 32 → 2`, predicts MNIST versus USPS, through a **gradient-reversal layer** (GRL).

The reconstruction makes this an autoencoder. The adversary distinguishes datasets; it does not match $z$ to a Gaussian prior.

Let $L_y$ be MNIST digit cross-entropy, $L_r$ the MNIST pixel mean-squared reconstruction error, and $L_d$ the mean of the two domain cross-entropies, weighted equally across domains.

| Component | Objective it minimizes |
|---|---|
| Encoder | $L_y + \beta L_r - \lambda L_d$ |
| Digit classifier | $L_y$ |
| Decoder | $\beta L_r$ |
| Domain classifier | $L_d$ |

The GRL is the identity during the forward pass and multiplies the gradient by $-\lambda$ during backpropagation **towards the encoder only**. The domain classifier still learns to distinguish domains normally. In the code, we add `domain_loss` with a positive sign; the GRL handles the sign and scaling once.

Both experiments reconstruct **MNIST only**. Target images enter the adapted experiment only through the domain branch, so target reconstruction is not a second changing factor. Reconstruction encourages retaining information, whereas domain confusion discourages retaining domain-specific information: these objectives can compete.

The source-only model has no domain loss. In the adapted run, $\lambda$ is zero for the first three epochs, then smoothly ramps towards `cfg.adversarial_weight`. The domain head can learn during this warm-up while its gradient into the encoder is zero. Both runs start from scratch with the same seed; adaptation does not receive extra source pretraining epochs.


In [ ]:
architecture = Autoencoder()
print(architecture)
print(f"Parameters: {sum(p.numel() for p in architecture.parameters()):,}")
del architecture


## 4. Source-only learning

Train the encoder, decoder and digit classifier on MNIST. USPS is not passed to this training call. A good MNIST classifier may perform worse on USPS because of domain shift.

We fit or load both models before revealing any target test labels. There is no early stopping, checkpoint selection or hyperparameter selection using USPS accuracy: the final epoch is saved.


In [ ]:
source_path = CHECKPOINT_DIR / "source_only.pt"
if MODE == "train":
    source_model, source_history = fit(source_train, None, cfg, adaptation=False, device=DEVICE)
    save_checkpoint(source_path, source_model, source_history, cfg, adaptation=False)
    source_config = asdict(cfg)
else:
    source_model, source_history, source_config = load_checkpoint(source_path, adaptation=False, device=DEVICE)


## 5. Add adversarial domain adaptation

Now give the domain classifier labelled **domain membership**: MNIST = 0, USPS = 1. Knowing which dataset an image came from is different from knowing its digit identity. USPS digit labels are unavailable to `fit`.

The encoder is encouraged to preserve MNIST class information while making the two domains harder to distinguish. This aligns marginal feature distributions; it does **not** explicitly pair source and target images of the same digit.


In [ ]:
adapted_path = CHECKPOINT_DIR / "adapted.pt"
if MODE == "train":
    adapted_model, adapted_history = fit(source_train, target_train, cfg, adaptation=True, device=DEVICE)
    save_checkpoint(adapted_path, adapted_model, adapted_history, cfg, adaptation=True)
    adapted_config = asdict(cfg)
else:
    adapted_model, adapted_history, adapted_config = load_checkpoint(adapted_path, adaptation=True, device=DEVICE)

if source_config != adapted_config:
    raise ValueError("The checkpoints use different training settings. Use a matched pair for this comparison.")


## 6. Training diagnostics

The source losses describe learning on MNIST. Balanced domain accuracy near 50% is consistent with confusion, but is not proof of invariance: the discriminator may simply be weak, and different digit classes may overlap incorrectly. This is training-domain accuracy of the jointly trained head, not an independent probe.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
for name, history in (("Source only", source_history), ("Adapted", adapted_history)):
    epochs = np.arange(1, len(history) + 1)
    axes[0].plot(epochs, [h["classification"] for h in history], label=name)
    axes[1].plot(epochs, [h["reconstruction"] for h in history], label=name)
axes[2].plot(np.arange(1, len(adapted_history) + 1),
             [h["domain_accuracy"] for h in adapted_history], label="Adapted")
axes[2].axhline(0.5, color="gray", linestyle="--", label="Chance (balanced)")
axes[2].set_ylim(0, 1)
for ax, title in zip(axes, ("MNIST digit cross-entropy", "MNIST reconstruction MSE", "Balanced domain accuracy")):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.show()


## 7. Reveal held-out labels: did transfer work?

**Training is finished.** Now load the official MNIST and USPS test splits. Neither test images nor test labels were used for fitting. USPS labels below are used only for scoring and colouring evaluation plots.

First inspect the source-only baseline. The desired pattern is MNIST accuracy substantially exceeding USPS accuracy; the code reports whatever actually happens. The two digit-coloured panels help distinguish useful class alignment from simple domain mixing.


In [ ]:
source_test, target_test = test_data(DATA_DIR)
source_results = {
    "MNIST": evaluate(source_model, source_test, DEVICE),
    "USPS": evaluate(source_model, target_test, DEVICE),
}
print("SOURCE ONLY")
for domain, result in source_results.items():
    print(f"{domain}: {result['accuracy']:.1%} accuracy (n={len(result['y']):,})")
plot_latent(source_results["MNIST"], source_results["USPS"], "Source-only learning")
plt.show()


### Evaluate the adapted model

The desired outcome is an increase in USPS accuracy while preserving useful MNIST classification. Domain overlap by itself is insufficient: compare the true digit colours and the measured accuracies.

Latent axes are shared within each figure, but the two separately trained models may rotate, reflect or rescale their coordinates. Do not interpret an absolute coordinate change between models as a physical change.


In [ ]:
adapted_results = {
    "MNIST": evaluate(adapted_model, source_test, DEVICE),
    "USPS": evaluate(adapted_model, target_test, DEVICE),
}
print("ADAPTED")
for domain, result in adapted_results.items():
    print(f"{domain}: {result['accuracy']:.1%} accuracy (n={len(result['y']):,})")
plot_latent(adapted_results["MNIST"], adapted_results["USPS"], "Adversarial adaptation")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
x = np.arange(2)
for offset, name, results in ((-0.18, "Source only", source_results), (0.18, "Adapted", adapted_results)):
    bars = ax.bar(x + offset, [100 * results[d]["accuracy"] for d in ("MNIST", "USPS")],
                  width=0.36, label=name)
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_xticks(x, ["MNIST test", "USPS test"])
ax.set_ylim(0, 110)
ax.set_ylabel("Digit accuracy (%)")
ax.legend()
plt.show()
change = 100 * (adapted_results["USPS"]["accuracy"] - source_results["USPS"]["accuracy"])
print(f"USPS change after adaptation: {change:+.1f} percentage points")


### The autoencoder branch

Reconstructions make the two-dimensional information constraint visible. The decoder was trained only on MNIST; USPS reconstruction is an additional qualitative diagnostic, not an optimized target task or the transfer success metric. Blurry outputs are plausible with only two latent coordinates.


In [ ]:
plot_reconstructions(adapted_model, source_test, target_test, DEVICE)
plt.show()


## 8. What this teaches about PBSHM

- Source labels teach a shared task; unlabelled target data can help adapt its representation.
- The adversary removes some domain-discriminating information, but cannot guarantee correct class alignment.
- A two-dimensional bottleneck is useful for teaching and deliberately restrictive for classification and reconstruction.
- Domain alignment can hurt if it mixes classes or discards useful information. Reconstruction and invariance can also compete. A small source–target gap or a negative transfer result is a valid finding, not a plotting error.

**For your lecture:** train in advance, inspect the measured result, and keep the matched pair of checkpoints. If performance is weak, inspect source loss / accuracy and class overlap before drawing a conclusion about transfer. More source training, a smaller reconstruction weight, or gentler adversarial pressure are reasonable experiments, but choosing settings after inspecting USPS labels turns those labels into validation information. Report that honestly; an unbiased new evaluation would then need an untouched target holdout. Do not present a selected favourable seed as a general guarantee.

**References**

- Ganin et al. (2016), [Domain-Adversarial Training of Neural Networks](https://jmlr.org/papers/v17/15-239.html): labelled-source / unlabelled-target learning with gradient reversal. This notebook adds a small reconstruction branch; it is not a reproduction of their benchmark results.
- Torchvision: [MNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html) and [USPS](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.USPS.html).

All implementation details are in the adjacent `demo.py`; only NumPy, PyTorch, torchvision and Matplotlib are needed by the demonstration itself.
